In [1]:
import heapq
from typing import List, Optional


class MedianFinder:
    """
    双堆求中位数数据结构

    思路：
    - 大顶堆（max_heap）：存储较小的一半元素（实际存储负数模拟）
    - 小顶堆（min_heap）：存储较大的一半元素
    - 保持：len(max_heap) >= len(min_heap) 且两者大小差不超过1
    - 这样中位数就是大顶堆的堆顶（奇数个元素时）或两个堆顶的平均值（偶数个时）
    """

    def __init__(self):
        # 大顶堆：存储较小的一半元素（存储负数来模拟大顶堆）
        self.max_heap: List[int] = []
        # 小顶堆：存储较大的一半元素
        self.min_heap: List[int] = []

        # 记录操作历史
        self.history: List[int] = []

    def addNum(self, num: int, verbose: bool = True) -> None:
        """添加一个整数"""
        self.history.append(num)

        if verbose:
            print(f"\n{'─'*50}")
            print(f"添加数字: {num}")
            print(f"当前所有数字: {self.history}")

        # 第一步：决定插入到哪个堆
        # 如果大顶堆为空或新数字小于等于大顶堆的堆顶（即最大值），则插入大顶堆
        if not self.max_heap or num <= -self.max_heap[0]:
            heapq.heappush(self.max_heap, -num)
            if verbose:
                print(f"  -> 插入到大顶堆（较小的一半）")
        else:
            heapq.heappush(self.min_heap, num)
            if verbose:
                print(f"  -> 插入到小顶堆（较大的一半）")

        # 第二步：平衡两个堆的大小
        # 确保大顶堆的大小 >= 小顶堆的大小，且两者大小差不超过1
        if len(self.max_heap) > len(self.min_heap) + 1:
            # 大顶堆太大了，移动一个元素到小顶堆
            moved = -heapq.heappop(self.max_heap)
            heapq.heappush(self.min_heap, moved)
            if verbose:
                print(f"  -> 平衡: 移动 {moved} 从大顶堆到小顶堆")
        elif len(self.min_heap) > len(self.max_heap):
            # 小顶堆太大了，移动一个元素到大顶堆
            moved = heapq.heappop(self.min_heap)
            heapq.heappush(self.max_heap, -moved)
            if verbose:
                print(f"  -> 平衡: 移动 {moved} 从小顶堆到大顶堆")

        if verbose:
            self.print_status()
            median = self.findMedian(verbose=False)
            print(f"  -> 当前中位数: {median}")

    def findMedian(self, verbose: bool = True) -> Optional[float]:
        """返回当前所有数字的中位数"""
        if not self.max_heap:
            return None

        if len(self.max_heap) > len(self.min_heap):
            # 奇数个元素，中位数是大顶堆的堆顶
            median = -self.max_heap[0]
        else:
            # 偶数个元素，中位数是两个堆顶的平均值
            median = (-self.max_heap[0] + self.min_heap[0]) / 2.0

        if verbose:
            print(f"中位数: {median}")
        return median

    def print_status(self) -> None:
        """打印当前堆的状态"""
        # 获取实际值
        max_heap_vals = sorted([-x for x in self.max_heap], reverse=True)
        min_heap_vals = sorted(self.min_heap)

        print(f"\n当前堆状态:")
        print(f"  大顶堆 (较小的一半): {max_heap_vals}  (堆顶: {max_heap_vals[0] if max_heap_vals else 'None'})")
        print(f"  小顶堆 (较大的一半): {min_heap_vals}  (堆顶: {min_heap_vals[0] if min_heap_vals else 'None'})")
        print(f"  大小: 大顶堆={len(self.max_heap)}, 小顶堆={len(self.min_heap)}")

    def get_all_numbers_sorted(self) -> List[int]:
        """获取所有数字的排序列表（用于验证）"""
        return sorted(self.history)

    def visualize_heap_structure(self) -> str:
        """可视化堆的结构"""
        result = []
        result.append("=" * 60)
        result.append("堆结构可视化")
        result.append("=" * 60)

        # 大顶堆可视化
        max_heap_vals = [-x for x in self.max_heap]
        result.append("\n【大顶堆 - 存储较小的一半】")
        if max_heap_vals:
            result.append(f"  堆顶（最大值）: {max(max_heap_vals)}")
            result.append(f"  完整数组: {max_heap_vals}")
            result.append(f"  树形结构:")
            result.append(self._heap_to_tree(max_heap_vals, is_max_heap=True))
        else:
            result.append("  空堆")

        # 小顶堆可视化
        result.append("\n【小顶堆 - 存储较大的一半】")
        if self.min_heap:
            result.append(f"  堆顶（最小值）: {min(self.min_heap)}")
            result.append(f"  完整数组: {self.min_heap}")
            result.append(f"  树形结构:")
            result.append(self._heap_to_tree(self.min_heap, is_max_heap=False))
        else:
            result.append("  空堆")

        return "\n".join(result)

    def _heap_to_tree(self, heap: List[int], is_max_heap: bool = False, index: int = 0, prefix: str = "") -> str:
        """将堆数组转换为树形文本表示"""
        if not heap or index >= len(heap):
            return ""

        lines = []
        val = heap[index]

        # 确定左右子节点索引
        left_idx = 2 * index + 1
        right_idx = 2 * index + 2

        # 当前节点
        lines.append(f"{prefix}└── {val}")

        # 递归子节点
        child_prefix = prefix + "    "
        if left_idx < len(heap) or right_idx < len(heap):
            if left_idx < len(heap):
                lines.append(self._heap_to_tree(heap, is_max_heap, left_idx, child_prefix))
            if right_idx < len(heap):
                lines.append(self._heap_to_tree(heap, is_max_heap, right_idx, child_prefix))

        return "\n".join(lines)


# ==================== 测试代码 ====================
def main():
    print("=" * 70)
    print("双堆求中位数 MedianFinder")
    print("=" * 70)

    # 创建 MedianFinder 实例
    mf = MedianFinder()

    # 测试序列
    test_sequence = [3, 1, 4, 1, 5]

    print("\n测试序列:", test_sequence)
    print("\n" + "=" * 70)
    print("逐步操作")
    print("=" * 70)

    for i, num in enumerate(test_sequence, 1):
        mf.addNum(num)

    print("\n" + "=" * 70)
    print("最终结果验证")
    print("=" * 70)

    # 显示最终堆结构
    print(mf.visualize_heap_structure())

    # 验证每个步骤的中位数
    print("\n" + "=" * 70)
    print("步骤验证")
    print("=" * 70)

    # 重新运行一次，记录每一步的中位数
    mf2 = MedianFinder()
    expected_results = [
        (3, 3.0),      # 添加3后中位数为3
        (1, 2.0),      # 添加1后中位数为2
        (4, 3.0),      # 添加4后中位数为3
        (1, 1.5),      # 添加1后中位数为1.5
        (5, 3.0)       # 添加5后中位数为3
    ]

    print("\n逐步验证:")
    print("-" * 50)
    print("步骤 | 添加数字 | 当前所有数字 | 中位数 | 预期 | 状态")
    print("-" * 50)

    all_numbers = []
    for i, (num, expected) in enumerate(expected_results, 1):
        mf2.addNum(num, verbose=False)
        all_numbers.append(num)
        median = mf2.findMedian(verbose=False)
        status = "✓" if median == expected else "✗"
        print(f"  {i}   |    {num}     | {str(all_numbers):<12} | {median} | {expected} | {status}")

    # 完整的中位数计算验证
    print("\n" + "=" * 70)
    print("详细验证")
    print("=" * 70)

    mf3 = MedianFinder()
    all_nums = []

    for i, num in enumerate([3, 1, 4, 1, 5], 1):
        all_nums.append(num)
        mf3.addNum(num, verbose=True)

        # 计算理论中位数进行验证
        sorted_nums = sorted(all_nums)
        n = len(sorted_nums)
        if n % 2 == 1:
            theoretical_median = sorted_nums[n // 2]
        else:
            theoretical_median = (sorted_nums[n // 2 - 1] + sorted_nums[n // 2]) / 2.0

        actual_median = mf3.findMedian(verbose=False)
        print(f"\n验证: 排序后={sorted_nums}, 理论中位数={theoretical_median}, 实际中位数={actual_median}")
        print(f"结果: {'✓ 正确' if actual_median == theoretical_median else '✗ 错误'}")
        print("=" * 50)

    # 复杂度分析
    print("\n" + "=" * 70)
    print("复杂度分析")
    print("=" * 70)
    print("""
┌─────────────────────────────────────────────────────────────────────┐
│ 操作          │ 时间复杂度 │ 空间复杂度 │ 说明                         │
├─────────────────────────────────────────────────────────────────────┤
│ addNum(num)   │ O(log n)   │ O(1)      │ 堆的插入操作是 O(log n)      │
│ findMedian()  │ O(1)       │ O(1)      │ 直接返回堆顶元素，O(1)时间    │
│ 总体          │ O(log n)   │ O(n)      │ n 是当前元素个数              │
└─────────────────────────────────────────────────────────────────────┘

详细分析：
1. addNum() 操作：
   - 每次插入到其中一个堆：O(log n)
   - 可能需要进行一次堆的平衡操作：O(log n)
   - 总计：O(log n)

2. findMedian() 操作：
   - 直接读取堆顶元素：O(1)
   - 如果是偶数个元素，计算平均值：O(1)

3. 空间复杂度：
   - 存储所有 n 个元素：O(n)
   - 每个元素只存储一次

优点：
- 支持动态添加数据
- 插入和查询效率都很高
- 不需要对所有数据进行排序

应用场景：
- 实时数据流的中位数监控
- 滑动窗口的中位数计算
- 数据分位数查询
    """)

    # 演示更多数据
    print("\n" + "=" * 70)
    print("更多数据演示")
    print("=" * 70)

    mf4 = MedianFinder()
    test_data = [10, 20, 30, 5, 15, 25, 35, 1]

    print("插入序列:", test_data)
    print("\n插入过程:")

    for num in test_data:
        mf4.addNum(num, verbose=True)

    print("\n最终结果:")
    print(f"  所有数字排序: {sorted(mf4.history)}")
    print(f"  中位数: {mf4.findMedian()}")


if __name__ == "__main__":
    main()

双堆求中位数 MedianFinder

测试序列: [3, 1, 4, 1, 5]

逐步操作

──────────────────────────────────────────────────
添加数字: 3
当前所有数字: [3]
  -> 插入到大顶堆（较小的一半）

当前堆状态:
  大顶堆 (较小的一半): [3]  (堆顶: 3)
  小顶堆 (较大的一半): []  (堆顶: None)
  大小: 大顶堆=1, 小顶堆=0
  -> 当前中位数: 3

──────────────────────────────────────────────────
添加数字: 1
当前所有数字: [3, 1]
  -> 插入到大顶堆（较小的一半）
  -> 平衡: 移动 3 从大顶堆到小顶堆

当前堆状态:
  大顶堆 (较小的一半): [1]  (堆顶: 1)
  小顶堆 (较大的一半): [3]  (堆顶: 3)
  大小: 大顶堆=1, 小顶堆=1
  -> 当前中位数: 2.0

──────────────────────────────────────────────────
添加数字: 4
当前所有数字: [3, 1, 4]
  -> 插入到小顶堆（较大的一半）
  -> 平衡: 移动 3 从小顶堆到大顶堆

当前堆状态:
  大顶堆 (较小的一半): [3, 1]  (堆顶: 3)
  小顶堆 (较大的一半): [4]  (堆顶: 4)
  大小: 大顶堆=2, 小顶堆=1
  -> 当前中位数: 3

──────────────────────────────────────────────────
添加数字: 1
当前所有数字: [3, 1, 4, 1]
  -> 插入到大顶堆（较小的一半）
  -> 平衡: 移动 3 从大顶堆到小顶堆

当前堆状态:
  大顶堆 (较小的一半): [1, 1]  (堆顶: 1)
  小顶堆 (较大的一半): [3, 4]  (堆顶: 3)
  大小: 大顶堆=2, 小顶堆=2
  -> 当前中位数: 2.0

──────────────────────────────────────────────────
添加数字: 5
当前所有数字: [3, 1, 4, 1, 5]
  -> 插入到小顶堆（较